In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
from matplotlib.ticker import FuncFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

from matplotlib.colors import LogNorm
# from scipy.interpolate import interp1d  
from scipy import stats
from matplotlib.ticker import LogLocator
from matplotlib.backends.backend_pdf import PdfPages

import pandas as pd

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def LimitTrackedArraysRows(trackedArrays, limit=None): #limit=(0,70000)
    if limit is None:
        return trackedArrays
    for parcelType in trackedArrays:
        for parcelDepth in trackedArrays[parcelType]:
            trackedArrays[parcelType][parcelDepth] \
            = trackedArrays[parcelType][parcelDepth][limit[0]:limit[1], :]
    return trackedArrays

In [ ]:
def GetData(tRange, pValues=np.arange(10000)):
    t1 = calculate_timestep(tRange[0])
    t2 = calculate_timestep(tRange[1])

    varNames = ['Z','Y','X'] + ['QCQI','QV','W']
    dtypes = {'Z': np.int16, 
              'Y': np.int16, 
              'X': np.int16, 
              'QCQI': np.float32, 
              'QV': np.float32,
               'W': np.float32}
    
    Nt = t2 - t1 + 1
    Np = len(pValues)
    dataDictionary = {var: np.zeros((Nt, Np), dtype=dtypes[var]) for var in varNames}
    dataDictionary['tSteps'] = np.arange(t1, t2+1)
    dataDictionary['pValues'] = pValues
    
    for i, t in enumerate(tqdm(dataDictionary['tSteps'])):
        for varName in varNames:
            raw = CallLagrangianArray(ModelData, DataManager, ModelData.timeStrings[t], varName)[pValues]
            dataDictionary[varName][i, :] = np.round(raw) if varName in ('Z','Y','X') else raw
            
    return dataDictionary

def calculate_timestep(time_hr=12):
    return np.abs(ModelData.time_hrs-time_hr).argmin()

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
#Find Detraining Parcels at MidLevels

def DetectDetrainment(dataDictionary, zLimit=(2,4), cloudThreshold=1e-5):
    zLevel = ModelData.zh[dataDictionary['Z']]
    QC = dataDictionary['QCQI']  # using qcqi for now

    inLayer = (zLevel >= zLimit[0]) & (zLevel <= zLimit[1])
    isCloudy = QC > cloudThreshold
    isEnvironment = ~isCloudy

    wasCloudy = np.zeros_like(isCloudy)
    wasCloudy[1:] = isCloudy[:-1]  # shift cloudy at t-1 boolean to time t

    movedGridbox = DetectGridboxCross(dataDictionary)

    isDetrain = wasCloudy & isEnvironment & inLayer & movedGridbox
    return isDetrain

def DetectGridboxCross(dataDictionary):
    X = dataDictionary['X']
    Y = dataDictionary['Y']

    xChanged = np.zeros_like(X, dtype=bool)
    yChanged = np.zeros_like(Y, dtype=bool)
    xChanged[1:] = X[1:] != X[:-1]
    yChanged[1:] = Y[1:] != Y[:-1]

    return xChanged | yChanged

In [ ]:
#Track Ascending Surface Parcels

def DetectUpdraftEvents(dataDictionary, wThresh=0.1,cloudThreshold=1e-5, 
                        minAscentTime_mins=20,minCloudTime_mins=5, 
                        minHeightGain=1.0, surfaceZLimit=0.5):
    W  = dataDictionary['W']
    QC = dataDictionary['QCQI']
    Z  = ModelData.zh[dataDictionary['Z']]
    Nt = Z.shape[0]
    
    ascentWindow = MinutesToTimesteps(minAscentTime_mins)
    minCloudSteps=MinutesToTimesteps(minCloudTime_mins)
    
    sustainedUpdraft = RollingAllTrue(W > wThresh, ascentWindow)          # w>thresh whole window
    cloudSteps = RollingCountTrue(QC > cloudThreshold, ascentWindow)      # cloudy for enough of it
    heightGain = Z[ascentWindow-1:] - Z[:Nt-ascentWindow+1]               # actually rose

    n = sustainedUpdraft.shape[0] # target length for the shifted checks below
    
    # only parcels that initiate below 0.5 km
    wasSurfaceParcel = np.zeros_like(sustainedUpdraft)
    wasSurfaceParcel[1:] = Z[:n-1] <= surfaceZLimit

    # make sure parcel undergoes acceleration (w failed threshold the step before)
    notUpdraftBefore = np.zeros_like(sustainedUpdraft)
    notUpdraftBefore[1:] = ~(W[:n-1] > wThresh)

    isUpdraftEvent_ascentRelative = (
        sustainedUpdraft
        & (cloudSteps >= minCloudSteps)
        & wasSurfaceParcel
        & notUpdraftBefore
        & (heightGain >= minHeightGain)
    )
    return isUpdraftEvent_ascentRelative   # shape (Nt-ascentWindow, Np); True at t = event starting at t

def MinutesToTimesteps(minutes):
    """Converts a duration in minutes to a number of model timesteps."""
    return max(1, round((minutes * 60) / ModelData.dt))

def RollingAllTrue(cond, window):
    """True at row t if cond is True for every step in t ... t + window-1."""
    c = np.cumsum(np.vstack([np.zeros((1, cond.shape[1])), cond]), axis=0)
    return (c[window:] - c[:-window]) == window

def RollingCountTrue(cond, window):
    """Count of True steps within t ... t + window-1."""
    c = np.cumsum(np.vstack([np.zeros((1, cond.shape[1])), cond]), axis=0)
    return c[window:] - c[:-window]

####################################################################################

def PadToFullNt(eventArray, Nt, fillValue=False):
    Np = eventArray.shape[1]
    nMissing = Nt - eventArray.shape[0]
    pad = np.full((nMissing, Np), fillValue, dtype=eventArray.dtype)
    return np.vstack([eventArray, pad])

####################################################################################
#Get Full-Duration Ascent Mask
def GetAscentDurationMask(dataDictionary, isUpdraftEvent, wThresh=0.1):
    """
    Given confirmed ascent-start flags (isUpdraftEvent) and the raw w>wThresh
    condition, find the true full-duration run each confirmed start belongs
    to, and return a (Nt, Np) mask that's True for the entire ascent span
    (not just the start timestep).
    """
    W = dataDictionary['W']
    Nt, Np = W.shape

    rawUpdraft = W > wThresh
    runStart, runEnd, runParcel = FindRuns(rawUpdraft)

    evT, evP = np.where(isUpdraftEvent)

    # match confirmed event starts to their corresponding raw run
    eventDF = pd.DataFrame({'t': evT, 'p': evP})
    runDF = pd.DataFrame({'t': runStart, 'end': runEnd, 'p': runParcel})
    matched = eventDF.merge(runDF, on=['t', 'p'], how='inner')

    isAscending = BuildFullDurationMask(
        matched['t'].values, matched['end'].values, matched['p'].values, Nt, Np
    )
    return isAscending

def FindRuns(condition):
    Nt, Np = condition.shape
    padded = np.zeros((Nt + 2, Np), dtype=np.int8)
    padded[1:-1, :] = condition
    d = np.diff(padded, axis=0)
    startRows, startCols = np.where(d == 1)
    endRows, endCols = np.where(d == -1)
    orderS = np.lexsort((startRows, startCols))
    orderE = np.lexsort((endRows, endCols))
    startRows, startCols = startRows[orderS], startCols[orderS]
    endRows = endRows[orderE]
    return startRows, endRows, startCols

def BuildFullDurationMask(startIdx, endIdx, parcelIdx, Nt, Np):
    delta = np.zeros((Nt + 1, Np), dtype=np.int8)
    np.add.at(delta, (startIdx, parcelIdx), 1)
    np.add.at(delta, (np.clip(endIdx, 0, Nt), parcelIdx), -1)
    return np.cumsum(delta[:-1], axis=0) > 0

####################################################################################

def PlotAscentValidation(dataDictionary, isAscending, p, wThresh=0.1, cloudThreshold=1e-5*1e3):
    W = dataDictionary['W'][:, p]
    QC = dataDictionary['QCQI'][:, p]*1e3
    tSteps = dataDictionary['tSteps']
    timeOfDay = ModelData.time_hrs[tSteps]

    fig, ax1 = plt.subplots(figsize=(10, 4))

    ax1.plot(timeOfDay, W, color='black', lw=1.5, label='w')
    ax1.axhline(wThresh, color='gray', ls='--', lw=1, label=f'wThresh={wThresh}')
    ax1.set_xlabel('time of day')
    ax1.set_ylabel('w (m/s)')
    ax1.xaxis.set_major_formatter(FuncFormatter(HoursToHHMM))

    mask = isAscending[:, p]

    # duration of the ascending region, from the actual masked timesteps
    if mask.any():
        durMin = round(mask.sum() * ModelData.dt / 60)
        ascentLabel = f'isAscending ({durMin} min)'
    else:
        ascentLabel = 'isAscending (none)'

    ax1.fill_between(timeOfDay, ax1.get_ylim()[0], ax1.get_ylim()[1], where=mask,
                      color='tab:orange', alpha=0.15, step='mid', label=ascentLabel)

    ax2 = ax1.twinx()
    ax2.plot(timeOfDay, QC, color='blue', lw=1.5, label='QCQI')
    ax2.axhline(cloudThreshold, color='blue', ls=':', lw=1, alpha=0.6,
                label=f'cloudThreshold={cloudThreshold}')
    ax2.set_ylabel('QCQI (cloud water+ice)', color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

    ax1.set_title(f'Parcel {p} — w and QCQI profile vs detected ascent')
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

def HoursToHHMM(hrs, pos=None):
    h = int(hrs)
    m = int(round((hrs - h) * 60))
    if m == 60:
        h += 1
        m = 0
    return f"{h:02d}:{m:02d}"

In [ ]:
#Find Ascending-Parcel Intersections with Detrainment Events

def FindIntersections(dataDictionary, isAscending, isDetrain, windowMin=30):
    cellId = AssignCellId(dataDictionary)
    Nt, Np = isAscending.shape
    windowSteps = MinutesToTimesteps(windowMin)

    # detrainment events: where and when each one happened
    dT, dP = np.where(isDetrain)
    dCell = cellId[dT, dP]

    # stretch each event forward so it stays "active" for windowMin minutes
    offsets = np.arange(windowSteps)
    expT = (dT[:, None] + offsets[None, :]).ravel()
    expCell = np.repeat(dCell, windowSteps)
    expDetrainParcel = np.repeat(dP, windowSteps)
    valid = expT < Nt
    detrainDF = pd.DataFrame({
        't': expT[valid], 'cellId': expCell[valid], 'detrain_parcel': expDetrainParcel[valid]
    })

    # ascending parcels: where each one is at every timestep
    aT, aP = np.where(isAscending)
    aCell = cellId[aT, aP]
    ascendDF = pd.DataFrame({'t': aT, 'cellId': aCell, 'ascend_parcel': aP})

    # match on same gridbox + same time
    matches = ascendDF.merge(detrainDF, on=['t', 'cellId'], how='inner')
    matches = matches[matches['ascend_parcel'] != matches['detrain_parcel']]  # drop self-matches

    return matches

def AssignCellId(dataDictionary):
    """
    Assign unique gridcell IDs 
    using linear bijective index mapping
    """
    dims = GetGridDims()
    X = dataDictionary['X']; Y = dataDictionary['Y']; Z = dataDictionary['Z']
    return np.ravel_multi_index((X, Y, Z), dims)

def InverseCellId(cellId):
    """
    Converts unique gridcell IDs u
    sing linear bijective index mapping 
    back into X, Y, Z coordinates.
    """
    dims = GetGridDims()
    return np.unravel_index(cellId, dims)

def GetGridDims():
    return (ModelData.Nxh + 1, ModelData.Nyh + 1, ModelData.Nzh + 1)
    
####################################################################################


import matplotlib.patheffects as pe

OUTLINE = [pe.withStroke(linewidth=3, foreground='white')]

def PlotIntersection(dataDictionary, intersections, row=0, dx_km=1.0,
                      xlim=(300, 320), ylim=(0, 5), ax=None, tOverride=None,
                      plotVar='w', plotType='contourf', nContours=15):
    cfg = VAR_CONFIG[plotVar]

    rec = intersections.iloc[row]
    tCenter = int(rec['t'])
    t = tCenter if tOverride is None else tOverride
    cellId = int(rec['cellId'])
    ascendP = int(rec['ascend_parcel'])
    detrainP = int(rec['detrain_parcel'])
    ix, iy, iz = InverseCellId(cellId)
    Nt = dataDictionary['Z'].shape[0]
    t = max(0, min(t, Nt - 1))
    tStepAbs = dataDictionary['tSteps'][t]

    field = CallVariable(ModelData, DataManager, ModelData.timeStrings[tStepAbs], cfg['varname'])
    fieldSlice = field[:, iy, :] * cfg['scale']

    nz, nx = fieldSlice.shape
    xGrid = np.arange(nx) * dx_km
    zGrid = ModelData.zh[:nz]

    ownFig = ax is None
    if ownFig:
        fig, ax = plt.subplots(figsize=(8, 5))

    if plotType == 'pcolormesh':
        im = ax.pcolormesh(xGrid, zGrid, fieldSlice, cmap=cfg['cmap'],
                            vmin=cfg['vmin'], vmax=cfg['vmax'], shading='auto')
    else:
        levels = np.linspace(cfg['vmin'], cfg['vmax'], nContours)
        im = ax.contourf(xGrid, zGrid, fieldSlice, levels=levels, cmap=cfg['cmap'], extend='both')

    xA = dataDictionary['X'][:t+1, ascendP] * dx_km
    zA = ModelData.zh[dataDictionary['Z'][:t+1, ascendP]]
    ax.plot(xA, zA, color='black', lw=1.5, marker='o', ms=3, label=f'ascending parcel {ascendP}',
            path_effects=OUTLINE)
    ax.scatter(xA[-1], zA[-1], color='lime', s=90, edgecolor='black', zorder=5,
               label='ascending parcel')

    dT_events, dP_events = np.where(isDetrain)
    detrainT = dT_events[(dP_events == detrainP)][0]
    tD = min(t, detrainT)
    xD = dataDictionary['X'][:tD+1, detrainP] * dx_km
    zD = ModelData.zh[dataDictionary['Z'][:tD+1, detrainP]]
    ax.plot(xD, zD, color='blue', lw=1.5, marker='s', ms=3, label=f'detraining parcel {detrainP}',
            path_effects=OUTLINE)
    ax.scatter(xD[-1], zD[-1], color='magenta', s=90, edgecolor='black', zorder=5,
               label='detrainment location')

    ax.set_xlabel('x (km)')
    ax.set_ylabel('z (km)')
    timeLabel = HoursToHHMM(ModelData.time_hrs[tStepAbs])
    tag = f't={timeLabel}' + (' (intersection)' if t == tCenter else '')
    ax.set_title(f'{plotVar} slice at y={iy*dx_km:.1f} km, {tag}')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    if ownFig:
        plt.colorbar(im, ax=ax, label=cfg['label'])
        ax.legend(fontsize=8, loc='best')
        plt.tight_layout()
        plt.show()
    return im

VAR_CONFIG = {
    'w':  {'varname': 'winterp', 'cmap': 'RdBu_r', 'vmin': -5,  'vmax': 5,   'label': 'w (m/s)',
           'scale': 1.0, 'maskBelow': None},
    'qc': {'varname': 'qcqi',    'cmap': 'turbo',  'vmin': 0.01, 'vmax': 2,  'label': 'QCQI (g/kg)',
           'scale': 1e3, 'maskBelow': 0.01},   # values below this become NaN -> transparent
}

def PlotIntersectionTimeSeries(dataDictionary, intersections, row=0, tOffsets=np.arange(-12,6,2),
                                 ncols=3, plotVar='w', **kwargs):
    cfg = VAR_CONFIG[plotVar]
    rec = intersections.iloc[row]
    tCenter = int(rec['t'])
    Nt = dataDictionary['Z'].shape[0]
    tList = [tCenter + off for off in tOffsets if 0 <= tCenter + off < Nt]
    nPanels = len(tList)
    nrows = int(np.ceil(nPanels / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 5*nrows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    im = None
    for ax, t in zip(axes, tList):
        im = PlotIntersection(dataDictionary, intersections, row=row, ax=ax, tOverride=t,
                               plotVar=plotVar, **kwargs)

    for ax in axes[nPanels:]:
        ax.axis('off')

    fig.colorbar(im, ax=axes[:nPanels], label=cfg['label'], shrink=0.6)
    axes[0].legend(fontsize=8, loc='best')
    plt.show()

In [ ]:
##############################################
#COMPUTING

In [ ]:
dataDictionary=GetData(tRange=[12,13],pValues=np.arange(20_000_000))

In [ ]:
#Find Detraining Parcels at MidLevels
isDetrain=DetectDetrainment(dataDictionary)

In [ ]:
#Track Ascending Surface Parcels
isUpdraftEvent_ascentRelative = DetectUpdraftEvents(dataDictionary)
isUpdraftEvent = PadToFullNt(isUpdraftEvent_ascentRelative, Nt=dataDictionary['Z'].shape[0])
isAscending = GetAscentDurationMask(dataDictionary, isUpdraftEvent)

# #testing
# examples = np.where(isAscending)[1]
# PlotAscentValidation(dataDictionary, isAscending, p=examples[5])

In [ ]:
#Find Ascending-Parcel Intersections with Detrainment Events
intersections = FindIntersections(dataDictionary, isAscending, isDetrain)
print(intersections)

# #testing
# PlotIntersectionTimeSeries(dataDictionary, intersections, row=0, plotVar='w')
# PlotIntersectionTimeSeries(dataDictionary, intersections, row=0, plotVar='qc')

In [ ]:
##############################################
#PLOTTING FUNCTIONS

In [ ]:
#Computing DCI Probabilities

def ComputeDeepConvectionProbability(dataDictionary, isAscending, intersections,
                                      deepThresh=6.0, shallowThresh=4.0):
    """
    deepThresh: height (km) above which a parcel counts as having gone deep
    shallowThresh: height (km) below which a parcel counts as having stayed shallow
    """
    Z = dataDictionary['Z']
    zLevel = ModelData.zh[Z]              # (Nt, Np) physical height for every parcel/timestep
    # top height each parcel reaches anywhere in the loaded window
    topHeight = zLevel.max(axis=0)        # (Np,)
    # all parcels that had at least one confirmed ascent event
    allAscendingParcels = np.unique(np.where(isAscending)[1])
    # subset that intersected a moistened detrainment event
    preconditionedParcels = intersections['ascend_parcel'].unique()
    # everyone else
    notPreconditionedParcels = np.setdiff1d(allAscendingParcels, preconditionedParcels)

    def ClassFraction(parcelIds, isMatch):
        if len(parcelIds) == 0:
            return np.nan, 0, 0
        match = isMatch[parcelIds]
        return match.mean(), match.sum(), len(parcelIds)

    isDeep = topHeight >= deepThresh
    isShallow = topHeight < shallowThresh

    pDeepGivenPrecond, nDeepPrecond, nPrecond = ClassFraction(preconditionedParcels, isDeep)
    pDeepGivenNotPrecond, nDeepNotPrecond, nNotPrecond = ClassFraction(notPreconditionedParcels, isDeep)

    pShallowGivenPrecond, nShallowPrecond, _ = ClassFraction(preconditionedParcels, isShallow)
    pShallowGivenNotPrecond, nShallowNotPrecond, _ = ClassFraction(notPreconditionedParcels, isShallow)

    print("--- Deep convection (top height >= {:.1f} km) ---".format(deepThresh))
    print(f"Preconditioned:     {nDeepPrecond} of {nPrecond} parcels went deep, P(Deep) = {pDeepGivenPrecond*100:.2f}%")
    print(f"Not preconditioned: {nDeepNotPrecond} of {nNotPrecond} parcels went deep, P(Deep) = {pDeepGivenNotPrecond*100:.2f}%")

    nDeepTotal = nDeepPrecond + nDeepNotPrecond
    fracDeepThatWerePrecond = nDeepPrecond / nDeepTotal if nDeepTotal > 0 else np.nan
    print(f"Of {nDeepTotal} total deep parcels, {nDeepPrecond} ({fracDeepThatWerePrecond*100:.1f}%) "
          f"underwent moist preconditioning")

    print("\n--- Stayed shallow (top height < {:.1f} km) ---".format(shallowThresh))
    print(f"Preconditioned:     {nShallowPrecond} of {nPrecond} parcels stayed shallow, P(Shallow) = {pShallowGivenPrecond*100:.2f}%")
    print(f"Not preconditioned: {nShallowNotPrecond} of {nNotPrecond} parcels stayed shallow, P(Shallow) = {pShallowGivenNotPrecond*100:.2f}%")

    nShallowTotal = nShallowPrecond + nShallowNotPrecond
    fracShallowThatWerePrecond = nShallowPrecond / nShallowTotal if nShallowTotal > 0 else np.nan
    print(f"Of {nShallowTotal} total shallow parcels, {nShallowPrecond} ({fracShallowThatWerePrecond*100:.1f}%) "
          f"underwent moist preconditioning")

    return {
        'pDeepGivenPrecond': pDeepGivenPrecond, 'pDeepGivenNotPrecond': pDeepGivenNotPrecond,
        'pShallowGivenPrecond': pShallowGivenPrecond, 'pShallowGivenNotPrecond': pShallowGivenNotPrecond,
        'nPrecond': nPrecond, 'nNotPrecond': nNotPrecond,
    }

def PlotProbabilityVsHeightThreshold(dataDictionary, isAscending, intersections,
                                       zRange=(1, 16), nSteps=40):
    Z = dataDictionary['Z']
    zLevel = ModelData.zh[Z]
    topHeight = zLevel.max(axis=0)

    allAscendingParcels = np.unique(np.where(isAscending)[1])
    preconditionedParcels = intersections['ascend_parcel'].unique()
    notPreconditionedParcels = np.setdiff1d(allAscendingParcels, preconditionedParcels)

    zThresholds = np.linspace(zRange[0], zRange[1], nSteps)

    # vectorized: for every threshold, fraction of each group with topHeight >= threshold
    topPrecond = topHeight[preconditionedParcels]
    topNotPrecond = topHeight[notPreconditionedParcels]

    pPrecond = np.array([(topPrecond >= z).mean() for z in zThresholds])
    pNotPrecond = np.array([(topNotPrecond >= z).mean() for z in zThresholds])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(zThresholds, pPrecond * 100, color='tab:orange', lw=2, label='Preconditioned')
    ax.plot(zThresholds, pNotPrecond * 100, color='tab:blue', lw=2, label='Not preconditioned')
    ax.set_xlabel('height threshold, z_top (km)')
    ax.set_ylabel('P(parcel reaches z_top) (%)')
    # ax.set_title('Probability of reaching height z_top vs. preconditioning status')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return zThresholds, pPrecond, pNotPrecond


In [ ]:
# Plot Property Histograms Relative to Intersection Time
def PlotPreconditioningHistograms(dataDictionary, intersections, varNames=['QV', 'W'],
                                    windowSteps=10, deepThresh=6.0, dt=None, normalize=True):
    isDeep_byParcel = isDeepByParcel(dataDictionary, deepThresh)
    histDF = BuildPreconditioningHistograms(dataDictionary, intersections,
                                             isDeep_byParcel, varNames, windowSteps)
    dt = ModelData.dt if dt is None else dt
    histDF['minutes'] = histDF['offset'] * dt / 60

    nRows = len(varNames)
    fig, axes = plt.subplots(nRows, 2, figsize=(11, 4.5*nRows))
    axes = np.atleast_2d(axes)
    plt.subplots_adjust(wspace=0.3, hspace=0.3, right=0.88)   # fixed layout, no tight_layout

    for i, var in enumerate(varNames):
        cfg = HIST_VAR_CONFIG.get(var, {'nBinsY': 40, 'maskBelow': None, 'yscale': 'linear'})
        varVals = histDF.loc[histDF['outcome'].isin(['shallow', 'deep']), var].copy()
        if cfg['maskBelow'] is not None:
            varVals = varVals[varVals >= cfg['maskBelow']]
        if len(varVals) == 0:
            continue
        yMin, yMax = varVals.min(), varVals.max()
        yBins = (np.logspace(np.log10(yMin), np.log10(yMax), cfg['nBinsY'])
                 if cfg['yscale'] == 'log' else
                 np.linspace(yMin, yMax, cfg['nBinsY']))

        rowData = {}
        for outcome in ['shallow', 'deep']:
            sub = histDF[histDF['outcome'] == outcome].copy()
            if cfg['maskBelow'] is not None:
                sub = sub[sub[var] >= cfg['maskBelow']]
            if len(sub) == 0:
                rowData[outcome] = None
                continue
            xBins = np.linspace(sub['minutes'].min(), sub['minutes'].max(), 2*windowSteps+1)
            counts, xEdges, yEdges = np.histogram2d(sub['minutes'], sub[var], bins=[xBins, yBins])
            if normalize:
                colSums = counts.sum(axis=1, keepdims=True)
                colSums[colSums == 0] = 1
                plotData = counts / colSums
            else:
                plotData = counts
            rowData[outcome] = {'data': plotData, 'xEdges': xEdges, 'yEdges': yEdges,
                                 'n': sub['parcel'].nunique()}

        validVmax = [rowData[o]['data'].max() for o in ['shallow', 'deep'] if rowData[o] is not None]
        vmax = max(validVmax) if validVmax else 1
        cbarLabel = 'fraction (per time-column)' if normalize else 'count'

        im = None
        for j, outcome in enumerate(['shallow', 'deep']):
            ax = axes[i, j]
            d = rowData[outcome]
            if d is None:
                ax.set_title(f'{var} — {outcome} (no data)')
                continue
            im = ax.pcolormesh(d['xEdges'], d['yEdges'], d['data'].T, cmap='viridis',
                                shading='auto', vmin=0, vmax=vmax)
            ax.axvline(0, color='white', ls='--', lw=1)
            ax.set_xlabel('minutes since intersection')
            ax.set_ylabel(var)
            if cfg['yscale'] == 'log':
                ax.set_yscale('log')
            ax.set_title(f'{var} — {outcome} (n={d["n"]} parcels)')

        # --- manually place a colorbar axes right next to the deep panel ---
        if im is not None:
            pos = axes[i, 1].get_position()   # figure-fraction bbox of the right panel
            gap = 0.01
            width = 0.015
            cax = fig.add_axes([pos.x1 + gap, pos.y0, width, pos.height])
            fig.colorbar(im, cax=cax, label=cbarLabel)

    plt.show()
    return histDF

HIST_VAR_CONFIG = {
    'W':    {'nBinsY': 40, 'maskBelow': None, 'yscale': 'linear'},
    'QV':   {'nBinsY': 40, 'maskBelow': None, 'yscale': 'linear'},
    'QCQI': {'nBinsY': 60, 'maskBelow': 1e-6, 'yscale': 'log'},
    'RH_vapor': {'nBinsY': 60, 'maskBelow': None, 'yscale': 'linear'},
}

def BuildPreconditioningHistograms(dataDictionary, intersections, isDeep_byParcel, varNames=['QV','W'], windowSteps=10):
    """
    For each intersection event, pull the ascending parcel's variable values
    for `windowSteps` timesteps before/after the intersection, tagged by
    whether that parcel eventually went deep or stayed shallow.
    """
    records = []
    for _, row in intersections.iterrows():
        p = int(row['ascend_parcel'])
        t0 = int(row['t'])
        outcome = 'deep' if isDeep_byParcel[p] else 'shallow'
        for offset in range(-windowSteps, windowSteps+1):
            t = t0 + offset
            if 0 <= t < dataDictionary['Z'].shape[0]:
                rec = {'parcel': p, 'offset': offset, 'outcome': outcome}
                for v in varNames:
                    rec[v] = dataDictionary[v][t, p]
                records.append(rec)
    return pd.DataFrame(records)

def isDeepByParcel(dataDictionary, deepThresh=6.0):
    """One boolean per parcel: did it ever reach deepThresh anywhere in the loaded window."""
    zLevel = ModelData.zh[dataDictionary['Z']]
    topHeight = zLevel.max(axis=0)
    return topHeight >= deepThresh

############################################################################################################################

def GetAscentStartByParcel(isUpdraftEvent):
    evT, evP = np.where(isUpdraftEvent)
    return dict(zip(evP.tolist(), evT.tolist()))

def BuildHistogramsFromAscentStart(dataDictionary, parcelIds, ascentStartByParcel,
                                     isDeep_byParcel, varNames=['QV','W'], windowSteps=10):
    """
    Same output shape as before, but t0 is always ascent start -- works
    identically for preconditioned and non-preconditioned parcels, so the
    two groups are on the same clock.
    """
    records = []
    Nt = dataDictionary['Z'].shape[0]
    for p in parcelIds:
        if p not in ascentStartByParcel:
            continue
        t0 = ascentStartByParcel[p]
        outcome = 'deep' if isDeep_byParcel[p] else 'shallow'
        for offset in range(-windowSteps, windowSteps+1):
            t = t0 + offset
            if 0 <= t < Nt:
                rec = {'parcel': p, 'offset': offset, 'outcome': outcome}
                for v in varNames:
                    rec[v] = dataDictionary[v][t, p]
                records.append(rec)
    return pd.DataFrame(records)

def PlotComparisonHistograms(dataDictionary, isAscending, isUpdraftEvent, intersections,
                               varNames=['QV','W'], windowSteps=10, deepThresh=6.0, dt=None):
    isDeep_byParcel = isDeepByParcel(dataDictionary, deepThresh)
    ascentStartByParcel = GetAscentStartByParcel(isUpdraftEvent)

    allAscendingParcels = np.unique(np.where(isAscending)[1])
    preconditionedParcels = intersections['ascend_parcel'].unique()
    notPreconditionedParcels = np.setdiff1d(allAscendingParcels, preconditionedParcels)

    histPrecond = BuildHistogramsFromAscentStart(dataDictionary, preconditionedParcels,
                                                   ascentStartByParcel, isDeep_byParcel, varNames, windowSteps)
    histPrecond['group'] = 'preconditioned'

    histNotPrecond = BuildHistogramsFromAscentStart(dataDictionary, notPreconditionedParcels,
                                                       ascentStartByParcel, isDeep_byParcel, varNames, windowSteps)
    histNotPrecond['group'] = 'not preconditioned'

    histAll = pd.concat([histPrecond, histNotPrecond], ignore_index=True)
    dt = ModelData.dt if dt is None else dt
    histAll['minutes'] = histAll['offset'] * dt / 60

    # mean time of intersection relative to ascent start, for the preconditioned group only
    interTimes = intersections.merge(
        pd.Series(ascentStartByParcel, name='ascentStart').rename_axis('ascend_parcel').reset_index(),
        on='ascend_parcel', how='left'
    )
    interTimes['offsetFromAscentStart_min'] = (interTimes['t'] - interTimes['ascentStart']) * dt / 60
    meanInterOffset = interTimes['offsetFromAscentStart_min'].mean()

    groupOutcomePairs = [('not preconditioned','shallow'), ('not preconditioned','deep'),
                          ('preconditioned','shallow'), ('preconditioned','deep')]

    for var in varNames:
        cfg = HIST_VAR_CONFIG.get(var, {'nBinsY': 40, 'maskBelow': None, 'yscale': 'linear'})
        varVals = histAll[var]
        if cfg['maskBelow'] is not None:
            varVals = varVals[varVals >= cfg['maskBelow']]
        yMin, yMax = varVals.min(), varVals.max()
        yBins = (np.logspace(np.log10(yMin), np.log10(yMax), cfg['nBinsY'])
                 if cfg['yscale']=='log' else np.linspace(yMin, yMax, cfg['nBinsY']))

        fig, axes = plt.subplots(2, 2, figsize=(11, 8))
        for ax, (grp, outcome) in zip(axes.ravel(), groupOutcomePairs):
            sub = histAll[(histAll['group']==grp) & (histAll['outcome']==outcome)].copy()
            if cfg['maskBelow'] is not None:
                sub = sub[sub[var] >= cfg['maskBelow']]
            if len(sub) == 0:
                ax.set_title(f'{grp} / {outcome} (no data)')
                continue
            xBins = np.linspace(sub['minutes'].min(), sub['minutes'].max(), 2*windowSteps+1)
            h = ax.hist2d(sub['minutes'], sub[var], bins=[xBins, yBins], cmap='viridis')
            plt.colorbar(h[3], ax=ax, label='count')
            ax.axvline(0, color='white', ls='--', lw=1, label='ascent start')
            if grp == 'preconditioned':
                ax.axvline(meanInterOffset, color='red', ls=':', lw=1.5, label='mean intersection time')
                ax.legend(fontsize=7, loc='upper right')
            if cfg['yscale']=='log': ax.set_yscale('log')
            ax.set_xlabel('minutes since ascent start')
            ax.set_ylabel(var)
            ax.set_title(f'{grp} / {outcome} (n={sub["parcel"].nunique()})')

        fig.suptitle(f'{var} — preconditioned vs. not preconditioned, shallow vs. deep')
        plt.tight_layout()
        plt.show()

    return histAll

In [ ]:
##############################################
#PLOTTING

In [ ]:
#Computing DCI Probabilities
results = ComputeDeepConvectionProbability(dataDictionary, isAscending, intersections)
zThresholds, pPrecond, pNotPrecond = PlotProbabilityVsHeightThreshold(dataDictionary, isAscending, intersections)

In [ ]:
# Plot Property Histograms Relative to Intersection Time
histDF = PlotPreconditioningHistograms(dataDictionary, intersections, varNames=['W','QV','QCQI'])

In [ ]:
def PlotComparisonHistograms(dataDictionary, isAscending, isUpdraftEvent, intersections,
                               varNames=['QV','W'], windowSteps=10, deepThresh=6.0, dt=None, normalize=True):
    isDeep_byParcel = isDeepByParcel(dataDictionary, deepThresh)
    ascentStartByParcel = GetAscentStartByParcel(isUpdraftEvent)

    allAscendingParcels = np.unique(np.where(isAscending)[1])
    preconditionedParcels = intersections['ascend_parcel'].unique()
    notPreconditionedParcels = np.setdiff1d(allAscendingParcels, preconditionedParcels)

    histPrecond = BuildHistogramsFromAscentStart(dataDictionary, preconditionedParcels,
                                                   ascentStartByParcel, isDeep_byParcel, varNames, windowSteps)
    histPrecond['group'] = 'preconditioned'

    histNotPrecond = BuildHistogramsFromAscentStart(dataDictionary, notPreconditionedParcels,
                                                       ascentStartByParcel, isDeep_byParcel, varNames, windowSteps)
    histNotPrecond['group'] = 'not preconditioned'

    histAll = pd.concat([histPrecond, histNotPrecond], ignore_index=True)
    dt = ModelData.dt if dt is None else dt
    histAll['minutes'] = histAll['offset'] * dt / 60

    interTimes = intersections.merge(
        pd.Series(ascentStartByParcel, name='ascentStart').rename_axis('ascend_parcel').reset_index(),
        on='ascend_parcel', how='left'
    )
    interTimes['offsetFromAscentStart_min'] = (interTimes['t'] - interTimes['ascentStart']) * dt / 60
    meanInterOffset = interTimes['offsetFromAscentStart_min'].mean()

    rowOrder = ['not preconditioned', 'preconditioned']   # top, bottom
    colOrder = ['shallow', 'deep']                         # left, right

    for var in varNames:
        cfg = HIST_VAR_CONFIG.get(var, {'nBinsY': 40, 'maskBelow': None, 'yscale': 'linear'})
        varVals = histAll[var]
        if cfg['maskBelow'] is not None:
            varVals = varVals[varVals >= cfg['maskBelow']]
        if len(varVals) == 0:
            continue
        yMin, yMax = varVals.min(), varVals.max()
        yBins = (np.logspace(np.log10(yMin), np.log10(yMax), cfg['nBinsY'])
                 if cfg['yscale']=='log' else np.linspace(yMin, yMax, cfg['nBinsY']))

        fig, axes = plt.subplots(2, 2, figsize=(11, 8))
        plt.subplots_adjust(wspace=0.35, hspace=0.4, right=0.88)

        # --- pass 1: compute plotData for all four panels first ---
        cellData = {}
        for gi, grp in enumerate(rowOrder):
            for ci, outcome in enumerate(colOrder):
                sub = histAll[(histAll['group']==grp) & (histAll['outcome']==outcome)].copy()
                if cfg['maskBelow'] is not None:
                    sub = sub[sub[var] >= cfg['maskBelow']]
                if len(sub) == 0:
                    cellData[(gi,ci)] = None
                    continue
                xBins = np.linspace(sub['minutes'].min(), sub['minutes'].max(), 2*windowSteps+1)
                counts, xEdges, yEdges = np.histogram2d(sub['minutes'], sub[var], bins=[xBins, yBins])
                if normalize:
                    colSums = counts.sum(axis=1, keepdims=True)
                    colSums[colSums == 0] = 1
                    plotData = counts / colSums
                else:
                    plotData = counts
                cellData[(gi,ci)] = {'data': plotData, 'xEdges': xEdges, 'yEdges': yEdges,
                                      'n': sub['parcel'].nunique()}

        cbarLabel = 'fraction (per time-column)' if normalize else 'count'

        # --- pass 2: plot, sharing vmax within each outcome COLUMN (both rows) ---
        for ci, outcome in enumerate(colOrder):
            validVmax = [cellData[(gi,ci)]['data'].max() for gi in range(2) if cellData[(gi,ci)] is not None]
            vmax = max(validVmax) if validVmax else 1

            im = None
            for gi, grp in enumerate(rowOrder):
                ax = axes[gi, ci]
                d = cellData[(gi,ci)]
                if d is None:
                    ax.set_title(f'{grp} / {outcome} (no data)')
                    continue
                im = ax.pcolormesh(d['xEdges'], d['yEdges'], d['data'].T, cmap='viridis',
                                    shading='auto', vmin=0, vmax=vmax)
                ax.axvline(0, color='white', ls='--', lw=1, label='ascent start')
                if grp == 'preconditioned':
                    ax.axvline(meanInterOffset, color='red', ls=':', lw=1.5, label='mean intersection time')
                    ax.legend(fontsize=7, loc='upper right')
                if cfg['yscale']=='log': ax.set_yscale('log')
                ax.set_xlabel('minutes since ascent start')
                ax.set_ylabel(var)
                ax.set_title(f'{grp} / {outcome} (n={d["n"]})')

            # one colorbar per column, spanning both rows (top + bottom axes)
            if im is not None:
                topPos = axes[0, ci].get_position()
                botPos = axes[1, ci].get_position()
                gap, width = 0.015, 0.015
                cax = fig.add_axes([topPos.x1 + gap, botPos.y0, width, topPos.y1 - botPos.y0])
                fig.colorbar(im, cax=cax, label=cbarLabel)

        fig.suptitle(f'{var} — preconditioned vs. not preconditioned, shallow vs. deep')
        plt.show()

    return histAll

In [ ]:
# Plot Property Histograms Relative to Intersection Time
histAll = PlotComparisonHistograms(dataDictionary, isAscending, isUpdraftEvent, intersections, varNames=['W','QV','QCQI'])